In [1]:
%load_ext autoreload
%autoreload 2
    
import re
import os
import sys
import random
import math
import matplotlib.pyplot as plt

from collections import Counter, defaultdict
from tqdm import tqdm_notebook
from glob import glob

sys.path.append("src")

from word_order.process_treebank import create_word_order_df, read_df


from multiblimp.languages import get_ud_langs

from word_order.create_pairs import create_pairs
from word_order.prediction_target import *
from word_order.process_treebank import load_treebank
from word_order.decision_tree import fit_dt
from word_order.entropy import order_entropy
from word_order.viz_tree import tree2html

import numpy as np

import pandas as pd


random.seed(42)
resource_dir = "/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/resources"

target = nmod_noun_target

deprel_dir = "_".join(target.child_deprels)
dt_df_dir = f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/dt_df/{deprel_dir}"
word_order_dir = f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/treebank_features/{deprel_dir}"


def get_impurity(n, min_n=300, max_n=4000, max_val=0.1, min_val=0.01):
    if n <= min_n:
        return max_val
    if n >= max_n:
        return min_val

    t = (math.log(n) - math.log(min_n)) / (math.log(max_n) - math.log(min_n))
    
    return max_val - t * (max_val - min_val)
    

langs = [path.split('/')[-1].split('.')[0] for path in glob(word_order_dir+"/*.csv")]

predictor_var = "deprel_order"

langs = ["Dutch"]
lang2data = {}


for lang in tqdm_notebook(sorted(langs), total=len(langs)):
    # lang = " ".join(fn.split("_")[:-2]) or fn
    html_file = f"word_order/decision_trees/html/{deprel_dir}/{lang}.html"

    # if lang in lang2data:
    #     continue
    
    print(lang)
    raw_df = read_df(lang, word_order_dir=word_order_dir)
    
    full_df = raw_df[raw_df[predictor_var].notnull()]

    if len(full_df) == 0:
        continue

    # subset core_arg df
    # if 'nsubj_sibling-deprel_aux' in full_df.columns:
    #     full_df = full_df[~full_df['nsubj_sibling-deprel_aux']]
    # if 'nsubj_sibling-deprel_cop' in full_df.columns:
    #     full_df = full_df[~full_df['nsubj_sibling-deprel_cop']]
    # if 'head_Tense' in full_df.columns:
    #     full_df = full_df[~full_df.head_Tense.isna()]

    omit_feats = {col for col in full_df.columns if ('form' in col) or ('lemma' in col)}
    omit_feats.add('core_args')

    min_impurity_decrease = get_impurity(len(full_df))
    
    model, dt_df, predictor_df = fit_dt(
        full_df, 
        target,
        verbose=1, 
        min_impurity_decrease=min_impurity_decrease,
        min_samples_leaf=10,
        save_to=os.path.join(dt_df_dir, lang),
        omit_feats=omit_feats,
    )

    if model is None:
        print("skipping", lang)
        continue
    
    # swap_df = create_pairs(
    #     dt_df,
    #     treebank, 
    #     swap_type="core_arg",
    #     save_to=os.path.join(dt_df_dir, f"{lang}.csv"),
    # )
    tree2html(
        model, 
        dt_df, 
        full_df, 
        predictor_var,
        target,
        html_file, 
        max_rows=15,
        meta={"Language": lang},
        only_show_real_orders=True,
        correlate_features=True,
    )

    lang2data[lang] = model, dt_df

/tmp/ipykernel_6170/854887390.py:63: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for lang in tqdm_notebook(sorted(langs), total=len(langs)):


  0%|          | 0/1 [00:00<?, ?it/s]

Dutch
Train acc 0.9827648438909944
Test acc  0.9853896103896104


/home/jaap/gpu/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4, 43, 44, 66, 90, 128, 145, 159, 169, 232, 235] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [13]:
dt_df.loc[10].sen

['De',
 'vroegere',
 'politiediensten',
 '(',
 'gemeentepolitie',
 ',',
 'gerechtelijke',
 'politie',
 ',',
 'rijkswacht',
 ',',
 '...',
 ')',
 'werden',
 'alle',
 'afgeschaft',
 '.']

In [27]:
from word_order.viz_tree import get_sample_ids

prep = model.named_steps["preprocessor"]
clf = model.named_steps["clf"]

predictor_value = 'Nn'

sample_ids = get_sample_ids(prep, clf, dt_df, predictor_var, max_rows=15)

# Pick the predictor_value where you saw the bug
# Check node 2
node_2_ids = sample_ids[predictor_value][2]  # adjust predictor_value accordingly

X_check = prep.transform(dt_df.loc[node_2_ids])
paths = clf.decision_path(X_check)
visits_node_2 = paths[:, 2].toarray().flatten().astype(bool)

print("All visit node 2?", visits_node_2.all())
print("Count that don't:", (~visits_node_2).sum())


All visit node 2? True
Count that don't: 0


In [36]:
for idx, ids in sample_ids['nN'].items():
    for jdx in ids:
        print(idx, dt_df.loc[jdx].leaf_id)

0 3
0 3
0 3
0 3
0 2
0 2
0 4
0 4
0 2
0 2
0 2
0 2
0 2
0 3
0 3
1 3
1 3
1 3
1 3
1 3
1 3
1 2
1 3
1 3
1 2
1 3
1 3
1 2
1 2
1 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
3 3
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4
4 4


In [44]:
for idx, row in dt_df.iterrows():
    # if row.sen == ['Dit', 'soort', 'verhalen', 'berusten', 'echter', 'zelden', 'op', 'bewijzen', '.']:
    if "duikbootmotor" in row.sen:
        print(row.sen, row.leaf_id, idx, row['nmod_child-feat_det_Definite'], full_df.loc[idx].nmod_form, full_df.loc[idx].head_form)
        # break


['Het', 'geval', 'van', 'dertig', 'meter', 'lengte', 'en', '300', 'ton', 'gewicht', '(', 'door', 'een', 'pantsering', 'van', '76', 'mm', 'rondom', ')', 'moest', 'voorzien', 'worden', 'van', 'drie', 'gevechtstorens', ',', 'ieder', 'bewapend', 'met', 'twee', '102', 'mm', 'kanonnen', 'en', 'voortgestuwd', 'worden', 'door', 'een', '800', 'pk', 'duikbootmotor', '.'] 4 5133 Ind pantsering ton
['Het', 'geval', 'van', 'dertig', 'meter', 'lengte', 'en', '300', 'ton', 'gewicht', '(', 'door', 'een', 'pantsering', 'van', '76', 'mm', 'rondom', ')', 'moest', 'voorzien', 'worden', 'van', 'drie', 'gevechtstorens', ',', 'ieder', 'bewapend', 'met', 'twee', '102', 'mm', 'kanonnen', 'en', 'voortgestuwd', 'worden', 'door', 'een', '800', 'pk', 'duikbootmotor', '.'] 4 5131 _missing meter geval
['Het', 'geval', 'van', 'dertig', 'meter', 'lengte', 'en', '300', 'ton', 'gewicht', '(', 'door', 'een', 'pantsering', 'van', '76', 'mm', 'rondom', ')', 'moest', 'voorzien', 'worden', 'van', 'drie', 'gevechtstorens', ',

In [34]:
dt_df.loc[idx].leaf_id

4

In [45]:
row['nmod_child-feat_det_Definite']

'Def'

In [6]:
from word_order.viz_deprel import generate_html_deprel_index


deprel = "_".join(target.child_deprels)
# deprel = "amod"

generate_html_deprel_index(
    f"/media/jaap/81b6ce8a-28e5-4eda-9c68-b13e0637cc4f/WORD_ORDER/dt_df/{deprel}", 
    f"word_order/decision_trees/html/{deprel}/",
    language_data=lang2data,
)

100%|█████████████████████████████████████████| 157/157 [01:15<00:00,  2.09it/s]


In [7]:
from word_order.viz_overview import generate_html_overview_index


generate_html_overview_index('word_order/decision_trees/html/')

In [17]:
from tqdm import tqdm


rows = []

for _, row in tqdm(full_df.iterrows()):
    for order in row.swap_order_candidates:
        rows.append((row.language, row.sen_str, row[f"{order}_sen_str"], row.core_args, order))

swap_df = pd.DataFrame(rows, columns=["language", "sen", "swap_sen", "sen_order", "swap_order"])
swap_df.to_csv("word_order/pairs/all_pairs.csv", index=False)

In [20]:
import pickle

with open("lang2data.pickle", "wb") as f:
    pickle.dump(lang2data, f)

full_df.to_csv("full_df.csv", index=False)

In [63]:
threshold = 0.1
swap_so = str.maketrans({"s": "o", "o": "s"})

correct_num_swaps = []
all_swap_order_candidates = []

all_orders = {"svo", "ovs", "osv", "sov", "vos", "vso"}

for _, row in dt_df.iterrows():
    core_arg = row.core_args
    swap_orders = all_orders - {core_arg, core_arg.translate(swap_so)}

    swap_order_candidates = [
        arg_order
        for arg_order in swap_orders
        if row[f"{arg_order}_entropy"] < threshold
    ]
    
    num_swaps = len(swap_order_candidates)
    
    correct_num_swaps.append(num_swaps)
    all_swap_order_candidates.append(swap_order_candidates)

dt_df["num_swaps"] = correct_num_swaps
dt_df["swap_order_candidates"] = all_swap_order_candidates

In [75]:
(swap_df["num_swaps"] > 0).mean()

0.0

In [49]:
row[[
    f"{arg_order}_entropy"
    for arg_order in swap_orders
]] < 0.04

vso_entropy    False
osv_entropy     True
sov_entropy     True
vos_entropy     True
Name: 996, dtype: bool

In [22]:
generate_html_index('word_order/decision_trees/html/')

In [24]:
"core_args" in raw_df.columns

False

In [37]:
"this Is a test".capitalize()

'This is a test'